In [21]:
import json
from molrl.dataloader import create_dataloader
from molrl.nnx_modules import SmilesEncoder, SmilesDecoder, PredictionHead
from molrl.models import EncoderPredictor, JointMolecularModel, SmilesAutoencoder
from molrl.training import oracle_train_step, oracle_val_step
from flax import nnx
from jax import numpy as jnp
import numpy as np
import optax
import orbax.checkpoint as ocp
from pathlib import Path
import random
import pandas as pd

In [ ]:
# --- Architecture configs (fixed from pretraining) ---
oracle_ckpt_dir = (Path.cwd() / "checkpoints/pretrained_oracle").resolve()
ae_ckpt_dir = (Path.cwd() / "../step_3_pretrain_autoencoder/checkpoints/pretrained_autoencoder").resolve()

with open(oracle_ckpt_dir.parent / "best_oracle_config.json") as f:
    oracle_config = json.load(f)

arch = {
    "vocab_size": 36,
    "max_seq_len": 102,
    "latent_dim": 128,
    "encoder_emb_dim": oracle_config["encoder_emb_dim"],
    "encoder_conv_channels": tuple(oracle_config["encoder_conv_channels"]),
    "encoder_kernel_sizes": tuple(oracle_config["encoder_kernel_sizes"]),
    "encoder_dropout_rate": oracle_config["encoder_dropout_rate"],
    "decoder_hidden_dim": 512,
    "decoder_emb_dim": 64,
    "prediction_head_hidden_dims": tuple(oracle_config["prediction_head_hidden_dim"] for _ in range(oracle_config["prediction_head_n_layers"])),
    "prediction_head_dropout_rate": oracle_config["prediction_head_dropout_rate"],
}


def build_pretrained_jmm() -> JointMolecularModel:
    """Build a JointMolecularModel and load pretrained weights from oracle + autoencoder checkpoints."""
    encoder = SmilesEncoder(
        vocab_size=arch["vocab_size"], latent_dim=arch["latent_dim"],
        emb_dim=arch["encoder_emb_dim"], conv_channels=arch["encoder_conv_channels"],
        kernel_sizes=arch["encoder_kernel_sizes"], dropout_rate=arch["encoder_dropout_rate"])

    decoder = SmilesDecoder(
        vocab_size=arch["vocab_size"], max_seq_len=arch["max_seq_len"],
        latent_dim=arch["latent_dim"], hidden_dim=arch["decoder_hidden_dim"],
        emb_dim=arch["decoder_emb_dim"])

    prediction_head = PredictionHead(
        latent_dim=arch["latent_dim"], hidden_dims=arch["prediction_head_hidden_dims"],
        dropout_rate=arch["prediction_head_dropout_rate"])

    jmm = JointMolecularModel(encoder, decoder, prediction_head)

    # restore oracle weights (encoder + prediction_head)
    checkpointer = ocp.PyTreeCheckpointer()
    oracle_template = EncoderPredictor(
        SmilesEncoder(vocab_size=arch["vocab_size"], latent_dim=arch["latent_dim"],
                      emb_dim=arch["encoder_emb_dim"], conv_channels=arch["encoder_conv_channels"],
                      kernel_sizes=arch["encoder_kernel_sizes"], dropout_rate=arch["encoder_dropout_rate"]),
        PredictionHead(latent_dim=arch["latent_dim"], hidden_dims=arch["prediction_head_hidden_dims"],
                       dropout_rate=arch["prediction_head_dropout_rate"]))
    oracle_state = checkpointer.restore(str(oracle_ckpt_dir), item=nnx.state(oracle_template))
    nnx.update(oracle_template, oracle_state)

    # restore AE weights (decoder)
    ae_template = SmilesAutoencoder(
        SmilesEncoder(vocab_size=arch["vocab_size"], latent_dim=arch["latent_dim"],
                      emb_dim=arch["encoder_emb_dim"], conv_channels=arch["encoder_conv_channels"],
                      kernel_sizes=arch["encoder_kernel_sizes"], dropout_rate=arch["encoder_dropout_rate"]),
        SmilesDecoder(vocab_size=arch["vocab_size"], max_seq_len=arch["max_seq_len"],
                      latent_dim=arch["latent_dim"], hidden_dim=arch["decoder_hidden_dim"],
                      emb_dim=arch["decoder_emb_dim"]))
    ae_state = checkpointer.restore(str(ae_ckpt_dir), item=nnx.state(ae_template))
    nnx.update(ae_template, ae_state)

    # transfer into JMM
    nnx.update(jmm.encoder, nnx.state(oracle_template.encoder))
    nnx.update(jmm.prediction_head, nnx.state(oracle_template.prediction_head))
    nnx.update(jmm.decoder, nnx.state(ae_template.decoder))

    return jmm


# quick test
jmm = build_pretrained_jmm()
print(f"JMM ready. Encoder + PredictionHead from oracle, Decoder from AE.")

JointMolecularModel initialized with pretrained weights:
  Encoder + PredictionHead from: /Users/derekvantilborg/Dropbox/coding/OOD-proof-mol-RL/project/step_4_train_oracle/checkpoints/pretrained_oracle
  Decoder from: /Users/derekvantilborg/Dropbox/coding/OOD-proof-mol-RL/project/step_3_pretrain_autoencoder/checkpoints/pretrained_autoencoder
